In [1]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv("../data/raw/layoffs.csv")
df_clean = df.copy()   # all cleaning is done on the copy
df_clean.shape

(4675, 11)

Convert date columns to real dates

In [3]:
df_clean['date'] = pd.to_datetime(df_clean['date'])
df_clean['date_added'] = pd.to_datetime(df_clean['date_added'])

df_clean[['date', 'date_added']].dtypes

date          datetime64[us]
date_added    datetime64[us]
dtype: object

Remove extra spaces from text columns

The columns are stored as the str dtype, so selecting include='object' can miss them.
Instead, is_string_dtype is used, which works for both object and str.
Stripping is done early so that every later check (unique values, counts) works on clean text.

In [4]:
text_columns = [c for c in df_clean.columns if pd.api.types.is_string_dtype(df_clean[c])]
print("Text columns found:", text_columns)

# count values with leading or trailing spaces BEFORE stripping
before = {c: int((df_clean[c].dropna() != df_clean[c].dropna().str.strip()).sum()) for c in text_columns}
print("Values with extra spaces before:", before)

for col in text_columns:
    df_clean[col] = df_clean[col].str.strip()

# verify AFTER stripping
after = {c: int((df_clean[c].dropna() != df_clean[c].dropna().str.strip()).sum()) for c in text_columns}
print("Values with extra spaces after: ", after)

Text columns found: ['company', 'location', 'industry', 'source', 'stage', 'country']
Values with extra spaces before: {'company': 16, 'location': 0, 'industry': 0, 'source': 3, 'stage': 0, 'country': 0}
Values with extra spaces after:  {'company': 0, 'location': 0, 'industry': 0, 'source': 0, 'stage': 0, 'country': 0}


**Result:** the cell above lists which text columns were checked and shows the count of values with extra spaces before and after stripping (the "after" counts must all be 0).

Industry

- 30 unique categories, with no spelling or capitalisation differences
- Only 2 missing values

In [5]:
df_clean['industry'].nunique(), df_clean['industry'].isnull().sum()

(30, np.int64(2))

In [6]:
# the 2 rows with missing industry
df_clean[df_clean['industry'].isnull()]

,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
536,Eyeo,"Cologne, Non-U.S.",NaN,2025-07-28,0.40,NaN,https://www.adexchanger.com/privacy/eyeo-names...,Unknown,NaN,Germany,2025-07-30
1604,Appsmith,SF Bay Area,35.0,2023-09-22,0.25,NaN,https://inc42.com/buzz/exclusive-insight-partn...,Series B,51.0,United States,2023-09-24


**Decision:** fill the 2 missing values with the **closest existing category**, not with new labels.

- The row that looked like an advertising company is set to `Marketing`
- The other row (a software company) has no matching category, so it is set to `Other`

**Reason:** labels like "Advertising" or "Software" would each create a category with a single row.
Single-row categories make charts messy and add nothing to the analysis.

In [7]:
df_clean.loc[536, 'industry'] = 'Marketing'
df_clean.loc[1604, 'industry'] = 'Other'

# verify
print("Missing industry values:", df_clean['industry'].isnull().sum())
df_clean.loc[[536, 1604], ['company', 'industry']]

Missing industry values: 0


,company,industry
536,Eyeo,Marketing
1604,Appsmith,Other


Country

In [8]:
# 'UAE' and 'United Arab Emirates' are the same country
df_clean['country'] = df_clean['country'].replace('UAE', 'United Arab Emirates')

print("Missing country values:", df_clean['country'].isnull().sum())
df_clean.loc[df_clean['country'].isnull(), ['company', 'location', 'country']]

Missing country values: 2


,company,location,country
1596,Fit Analytics,"Berlin, Non-U.S.",NaN
1882,Ludia,"Montreal, Non-U.S.",NaN


**Decision:** fill the 2 missing countries using each row's `location` column.

**Reason:** the location clearly names the country, so this is a fact taken from the same row, not a guess.

In [9]:
df_clean.loc[1596, 'country'] = 'Germany'
df_clean.loc[1882, 'country'] = 'Canada'

# verify
print("Missing country values:", df_clean['country'].isnull().sum())
df_clean.loc[[1596, 1882], ['company', 'location', 'country']]

Missing country values: 0


,company,location,country
1596,Fit Analytics,"Berlin, Non-U.S.",Germany
1882,Ludia,"Montreal, Non-U.S.",Canada


Stage

In [10]:
print("Missing stage values:", df_clean['stage'].isnull().sum())
df_clean[df_clean['stage'].isnull()][['company', 'stage']]

Missing stage values: 8


,company,stage
280,Zondacrypto,NaN
322,Flex AI,NaN
334,OP Labs,NaN
2531,Soundwide,NaN
3143,Advata,NaN
3149,Spreetail,NaN
3498,Gatherly,NaN
3881,Zapp,NaN


**Decision:** fill the 8 missing stage values with `Unknown`.

**Reason:** the dataset already uses `Unknown` as a category, so this keeps the data consistent.

In [11]:
df_clean['stage'] = df_clean['stage'].fillna('Unknown')

df_clean['stage'].isnull().sum()

np.int64(0)

Percentage laid off equal to 0

In [12]:
print("Rows with percentage_laid_off == 0:", (df_clean['percentage_laid_off'] == 0).sum())
df_clean[df_clean['percentage_laid_off'] == 0]

Rows with percentage_laid_off == 0: 1


,company,location,total_laid_off,date,percentage_laid_off,industry,source,stage,funds_raised,country,date_added
3732,TaskUs,Los Angeles,52.0,2022-06-21,0.0,Support,Internal memo,Post-IPO,279.0,United States,2022-06-22


**Observation:** only one record (TaskUs) has `percentage_laid_off = 0.0`, even though 52 employees were laid off.

**Decision:** keep the record unchanged.

**Reason:** the likely cause is rounding (52 people is a very small share of a large company). The actual percentage cannot be calculated because the dataset has no company headcount column.

Both layoff columns missing

In [13]:
both_missing = df_clean['total_laid_off'].isnull() & df_clean['percentage_laid_off'].isnull()

print("Rows with both missing:", both_missing.sum())
print("Share of dataset: ", round(both_missing.sum() / len(df_clean) * 100, 1), "%")

Rows with both missing: 763
Share of dataset:  16.3 %


**Observation:** 763 of 4,675 rows (16.3%) have neither `total_laid_off` nor `percentage_laid_off`.

**Decision:** keep all of them.

**Reason:** they still record that a layoff event happened (company, date, industry, country). Missing numbers are **not** filled with averages, because that would invent data.

**Additional step:** a True/False column `size_known` marks rows where the layoff count exists, so size-based analysis can filter to those rows only.

In [14]:
df_clean['size_known'] = df_clean['total_laid_off'].notnull()

df_clean['size_known'].value_counts()

size_known
True     3048
False    1627
Name: count, dtype: int64

Duplicates

A full-row duplicate check earlier found **0** exact duplicates.
A looser check on `company` + `date` can still find rows that look repeated, so each case is examined.

In [15]:
# every row that shares a company and date with another row
same_day = df_clean[
    df_clean.duplicated(subset=['company', 'date'], keep=False)
].sort_values(['company', 'date'])

print("Rows sharing the same company and date:", len(same_day))
same_day[['company', 'location', 'date', 'total_laid_off', 'percentage_laid_off', 'source']]

Rows sharing the same company and date: 44


,company,location,date,total_laid_off,percentage_laid_off,source
144,Amazon,Port St. Lucie,2026-07-22,1110.0,NaN,https://www.msn.com/en-us/money/companies/amaz...
146,Amazon,Seattle,2026-07-22,NaN,NaN,https://www.reuters.com/business/world-at-work...
3629,Anodot,"Tel Aviv, Non-U.S.",2022-07-06,35.0,0.27,https://www.calcalistech.com/ctechnews/article...
3631,Anodot,"Tel Aviv, Non-U.S.",2022-07-06,15.0,0.20,https://www.calcalistech.com/ctechnews/article...
3196,Beyond Meat,Los Angeles,2022-10-14,200.0,0.19,https://www.cnbc.com/2022/10/14/beyond-meat-to...
3197,Beyond Meat,Los Angeles,2022-10-14,200.0,0.19,https://www.fooddive.com/news/beyond-meat-layo...
4492,Bustle Digital Group,New York City,2020-04-03,24.0,0.08,https://digiday.com/media/bustle-digital-group...
4493,Bustle Digital Group,New York City,2020-04-03,19.0,NaN,https://www.adweek.com/media/bdg-input-mic-lay...
3738,Bybit,"Singapore, Non-U.S.",2022-06-20,600.0,0.30,https://techcrunch.com/2022/06/20/bybit-crypto...
3745,Bybit,"Singapore, Non-U.S.",2022-06-20,NaN,NaN,https://twitter.com/WuBlockchain/status/153880...


**Decision rule:**

- Same company, same date, **and** same location, layoff count and percentage: this is the **same event reported twice** (a true duplicate). Drop it and keep the first copy.
- Same company and date but a **different** location, count or percentage: these are **separate events** (for example, different offices or two announcements on one day). Keep them.

**Reason:** dropping every company-date repeat would delete real events, and keeping true repeats would count the same layoff twice.

In [16]:
dup_key = ['company', 'location', 'date', 'total_laid_off', 'percentage_laid_off']

true_duplicates = df_clean[df_clean.duplicated(subset=dup_key, keep=False)].sort_values(dup_key)
print("Rows involved in true duplicates:", len(true_duplicates))
true_duplicates[['company', 'location', 'date', 'total_laid_off', 'percentage_laid_off', 'source']]

Rows involved in true duplicates: 10


,company,location,date,total_laid_off,percentage_laid_off,source
3196,Beyond Meat,Los Angeles,2022-10-14,200.0,0.19,https://www.cnbc.com/2022/10/14/beyond-meat-to...
3197,Beyond Meat,Los Angeles,2022-10-14,200.0,0.19,https://www.fooddive.com/news/beyond-meat-layo...
3807,Cazoo,"London, Non-U.S.",2022-06-07,750.0,0.15,https://news.sky.com/story/car-retailer-cazoo-...
3808,Cazoo,"London, Non-U.S.",2022-06-07,750.0,0.15,https://sifted.eu/articles/cazoo-layoffs-europ...
1853,FNZ,"London, Non-U.S.",2023-07-06,1000.0,0.15,https://citywire.com/new-model-adviser/news/ex...
1854,FNZ,"London, Non-U.S.",2023-07-06,1000.0,0.15,https://citywire.com/new-model-adviser/news/ex...
3127,Oda,"Oslo, Non-U.S.",2022-11-01,70.0,0.18,https://www.dn.no/handel/oda/oda-kutter-70-ars...
3128,Oda,"Oslo, Non-U.S.",2022-11-01,70.0,0.18,https://shifter.no/nyheter/oda-sparker-70-og-h...
3867,Terminus,Atlanta,2022-05-27,NaN,NaN,https://www.bizjournals.com/atlanta/inno/stori...
3868,Terminus,Atlanta,2022-05-27,NaN,NaN,https://www.bizjournals.com/atlanta/inno/stori...


In [17]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=dup_key, keep='first').reset_index(drop=True)
rows_after = len(df_clean)

print("Rows before:", rows_before)
print("Rows after: ", rows_after)
print("Rows removed:", rows_before - rows_after)

Rows before: 4675
Rows after:  4670
Rows removed: 5


**Note:** the index is reset after dropping. The hard-coded index fixes in sections 4 and 5 were done **before** this step, so they used the original row labels correctly.

Create time columns

These support the yearly and monthly trend analysis in the next phase.

In [18]:
df_clean['year'] = df_clean['date'].dt.year
df_clean['month'] = df_clean['date'].dt.month

df_clean[['date', 'year', 'month']].head()

,date,year,month
0,2026-11-11,2026,11
1,2026-11-11,2026,11
2,2026-11-11,2026,11
3,2026-11-10,2026,11
4,2026-11-09,2026,11


Final checks

In [19]:
print("Shape:", df_clean.shape)
print()
print("Date range:", df_clean['date'].min().date(), "to", df_clean['date'].max().date())
print()
print("Missing values per column:")
df_clean.isnull().sum()

Shape: (4670, 14)

Date range: 2020-03-11 to 2026-11-11

Missing values per column:


company                   0
location                  1
total_laid_off         1626
date                      0
percentage_laid_off    1757
industry                  0
source                    3
stage                     0
funds_raised            556
country                   0
date_added                0
size_known                0
year                      0
month                     0
dtype: int64

In [20]:
# these columns must now have no missing values
df_clean[['industry', 'stage', 'country']].isnull().sum()

industry    0
stage       0
country     0
dtype: int64

In [21]:
df_clean.dtypes

company                           str
location                          str
total_laid_off                float64
date                   datetime64[us]
percentage_laid_off           float64
industry                          str
source                            str
stage                             str
funds_raised                  float64
country                           str
date_added             datetime64[us]
size_known                       bool
year                            int32
month                           int32
dtype: object

**Expected after cleaning:**

- `industry`, `stage`, `country`: 0 missing
- `total_laid_off`, `percentage_laid_off`, `funds_raised`: still have missing values on purpose (they are not invented)
- New columns present: `size_known`, `year`, `month`

Save the cleaned data

In [22]:
os.makedirs("../data/cleaned", exist_ok=True)

df_clean.to_csv("../data/cleaned/layoffs_cleaned.csv", index=False)
print("Saved:", df_clean.shape)

Saved: (4670, 14)


In [23]:
# checkpoint: reload the saved file and confirm it matches
check = pd.read_csv("../data/cleaned/layoffs_cleaned.csv")
print("Reloaded shape:", check.shape)
print("Same number of rows as df_clean:", len(check) == len(df_clean))

Reloaded shape: (4670, 14)
Same number of rows as df_clean: True
